# Скрипт для скачивания датасета FNSPID

Датасет **FNSPID** (Financial News and Stock Price Integration Dataset) размещён на HuggingFace и скачивается из оригинального ноутбука по базовому [URL](https://huggingface.co/datasets/Zihan1004/FNSPID)

Авторы датасета — Zihan Dong, Xinyu Fan, Zhiyuan Peng. Датасет покрывает период 1999 - 2023 гг. и объединяет новостные статьи, привязанные к тикерам американских компаний, с дневными ценовыми данными по тем же тикерам.

Датасет организован в две папки, и из каждой скачиваются свои файлы:

**`Stock_news/`** — новостной корпус. Содержит два CSV-файла:

- **`All_external.csv`** — основной файл с новостями. Каждая строка — одна статья: дата публикации, тикер компании, заголовок, URL, автор, а также четыре варианта автоматических саммари (LSA, Luhn, TextRank, LexRank). Это файл с более высоким заполнением текстовых полей, поэтому при дедупликации он используется как приоритетный источник.
- **`nasdaq_exteral_data.csv`** — вспомогательный файл с новостями из того же источника (NASDAQ). Структура колонок та же. Он частично дублирует `All_external.csv`, но содержит дополнительные записи. Именно поэтому в ноутбуке выполняется дедупликация по хешу комбинации (тикер + URL) или (тикер + дата + заголовок), и из суммарных 15,5 млн строк остаётся 1,23 млн уникальных статей.

**`Stock_price/`** — ценовые данные. Содержит один архив:

- **`full_history.zip`** - при распаковке создаёт папку `full_history/full_history/`, внутри которой лежит по одному CSV-файлу на каждый тикер (например, `TSLA.csv`, `AAPL.csv` и т.д.). Каждый файл содержит дневные данные: дата, цена открытия, закрытия, скорректированная цена закрытия (adjusted close) и объём торгов. Всего в архиве 7 691 тикер.

### Требования: pip install requests tqdm yfinance pandas

In [ ]:
import os
import time
import glob
import shutil
import zipfile

import pandas as pd
import requests
import yfinance as yf
from tqdm import tqdm

In [ ]:
BASE_URL = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main"

DOWNLOADS = {
    "All_external.csv":        "Stock_news/All_external.csv",
    "nasdaq_exteral_data.csv": "Stock_news/nasdaq_exteral_data.csv",
    "full_history.zip":        "Stock_price/full_history.zip",
}

OUTPUT_DIR = "outputs_01"
META_CACHE = os.path.join(OUTPUT_DIR, "ticker_meta_yahoo.csv")
PRICE_DIR = os.path.join("full_history", "full_history")

YAHOO_SLEEP = 0.25

In [ ]:
def download_file(url: str, filename: str):
    if os.path.exists(filename):
        os.remove(filename)

    print(f"  Скачиваю: {filename} ...")
    with requests.get(url, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        with open(filename, "wb") as f:
            with tqdm(total=total, unit="B", unit_scale=True) as pbar:
                for chunk in resp.iter_content(1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
    print(f"  Готово: {filename}")


def download_all():
    print("Шаг 1. Скачивание файлов с HuggingFace")
    for local_name, remote_path in DOWNLOADS.items():
        download_file(f"{BASE_URL}/{remote_path}", local_name)

In [ ]:
def extract_prices():
    print()
    print("Шаг 2. Распаковка ценовых данных")

    if os.path.isdir("full_history"):
        shutil.rmtree("full_history")

    print("Распаковываю full_history.zip ...")
    with zipfile.ZipFile("full_history.zip", "r") as z:
        z.extractall("full_history")

    n = len(glob.glob(os.path.join(PRICE_DIR, "*.csv")))
    print(f"Готово: {n} файлов в {PRICE_DIR}")

In [ ]:
def get_company_metadata(ticker: str) -> dict:
    symbol = str(ticker).replace(".", "-").strip()
    try:
        info = yf.Ticker(symbol).get_info()
        return {
            "ticker":       str(ticker),
            "yahoo_symbol": symbol,
            "company_name": info.get("longName") or info.get("shortName"),
            "sector":       info.get("sector"),
            "industry":     info.get("industry"),
            "exchange":     info.get("exchange"),
            "quoteType":    info.get("quoteType"),
        }
    except Exception:
        return {
            "ticker": str(ticker), "yahoo_symbol": symbol,
            "company_name": None, "sector": None, "industry": None,
            "exchange": None, "quoteType": None,
        }

In [ ]:
def fetch_yahoo_metadata(tickers: list):
    print()
    print("Шаг 3. Загрузка метаданных из Yahoo Finance")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if os.path.exists(META_CACHE):
        os.remove(META_CACHE)

    print(f"Загружаю метаданные для {len(tickers)} тикеров ...")

    rows = []
    for t in tqdm(tickers, desc="  Yahoo Finance"):
        rows.append(get_company_metadata(t))
        time.sleep(YAHOO_SLEEP)

    df = pd.DataFrame(rows)
    df.to_csv(META_CACHE, index=False)
    print(f"Сохранено: {META_CACHE} ({len(df)} тикеров)")

In [ ]:
if __name__ == "__main__":
    download_all()
    extract_prices()

    tickers = [
        os.path.splitext(os.path.basename(f))[0]
        for f in glob.glob(os.path.join(PRICE_DIR, "*.csv"))
    ]
    print(f"\nНайдено тикеров: {len(tickers)}")

    fetch_yahoo_metadata(tickers)

    print()
    print("Готово!")